# Agentic Workflows as Probabilistic Programs

LLMs are increasingly being used as agents when it comes to solving problems and tasks. These scenarios involve single or multiple LLMs interacting with the environment via tools and gathering information in order to accomplish a task or to find answer to a complex query. There are several workflows that people have come up with based on specific tasks and tools available for LLMs to use. Some notable examples include Self-Refine, Reflexion, ReAct, Magentic-One etc.

Since language models are generative, we envision these workflows to be probabilistic programs. With this perspective, we aim to separate the "what" from "how", the declarative specification of the task from how it is achieved. Now, which components of these workflows belong in "what" and which components belong in "how" is unfortunately subjective and the right choice depends on which separation opens the most uses. One perspective where everything other than the query and the final answer belongs in "how" is described in this [overleaf document](https://www.overleaf.com/read/vkkvczrnnkwb#e0fe1e)

In this note, we focus on a different perspective where a few more components than the query and the final answer make it to "what". This is primarily done to allow reasoning about a wider set of components of these agentic workflows.

## ReAct

A typical ReAct pipeline sends an LLM two prompts 

1) A system prompt that tells you what sequence of steps to follow 
2) A task prompt describing the query at hand. Upon receiving these prompts, the LLM goes through a loop of outputting thoughts and code blocks which are executed, appended to the prompt and the loop continues until final_answer is reached.

Algorithmically, it looks something like below:

```python
system_prompt = …
task = …
is_final = False
memory = task + system_prompt
while not is_final:
	thought, action = sample(LLM(.. | memory))
	is_final = check(action)
	if is_final:
		answer = extract_answer(action)
	else:
		observation = exec(action)
		memory += observation
return answer
```

The above code is currently implemented as CodeAgent in smolagents library.



This [notebook](playground/fun.ipynb) uses CodeAgent in smolagent library to carry out a ReAct loop on the first 20 examples of GAIA benchmark. It shows 20% accuracy which varies significantly from run to run and uses on average 229,556.85 tokens per question which is a lot since the limit of context window for OpenAI API is 128,000.

## Token discrepancies

It can be confusing in smolagents that how are the statistics about input and output tokens are being generated. Atleast for OpenAI model, these token statistics printed by the library are cumulative input and output tokens totaled over all the API calls made for a query till a particular point. This differs from the length of the context passed to the single call to API.

Also, note that OpenAI APIs perform [**prompt caching**](https://developers.openai.com/api/docs/guides/prompt-caching) which can affect how the price of the tokens differ.

## Baselines

Before we move on, let's see how different LLMs perform as backbones for ReAct. 

**Full validation set (165 questions):**

- The following list does not include deepseek-ai/DeepSeek-R1 because it is being deprecated and not available for serverless use on TogetherAI.
- The Qwen models are being used with `{"enable_thinking": False}` as a parameter.

In [13]:
import pickle
from pathlib import Path

import pandas as pd

BASELINE_DIR = Path("baseline")

# Read from the per-example .pkl cache dirs rather than the .jsonl logs: evaluate_agent
# writes pickle_dir/{i}.pkl once per example index, overwritten in place on recompute, so
# each file always reflects that example's single latest result -- no de-duplication needed,
# unlike the .jsonl logs which are append-only across every run/restart across sessions.
MODEL_DIRS = {
    "gpt-4o": BASELINE_DIR / "naive_react_gpt-4o_False",
    "gpt-5.4-mini": BASELINE_DIR / "naive_react_gpt-5.4-mini_False",
    "Qwen3.7-Plus": BASELINE_DIR / "naive_react_Qwen" / "Qwen3.7-Plus_False",
    "Qwen3.5-9B": BASELINE_DIR / "naive_react_Qwen" / "Qwen3.5-9B_False",
}


def summarize_results(pkl_dir: Path):
    records = []
    for pkl_file in sorted(pkl_dir.glob("*.pkl"), key=lambda p: int(p.stem)):
        with open(pkl_file, "rb") as f:
            row = pickle.load(f)
        token_counts = row.get("token_counts") or {}
        records.append({
            "is_correct": bool(row.get("is_correct", False)),
            "num_steps": row.get("num_steps", 0),
            "input_tokens": token_counts.get("input_tokens", 0),
            "output_tokens": token_counts.get("output_tokens", 0),
            "total_tokens": token_counts.get("total_tokens", 0),
            "error": row.get("error"),
        })

    n = len(records)
    if n == 0:
        return None
    correct = sum(r["is_correct"] for r in records)
    return {
        "n": n,
        "correct": correct,
        "accuracy": correct / n,
        "avg_steps": sum(r["num_steps"] for r in records) / n,
        "avg_input_tokens": sum(r["input_tokens"] for r in records) / n,
        "avg_output_tokens": sum(r["output_tokens"] for r in records) / n,
        "avg_tokens": sum(r["total_tokens"] for r in records) / n,
        "errors": sum(1 for r in records if r["error"]),
    }


rows = []
for model_name, pkl_dir in MODEL_DIRS.items():
    if not pkl_dir.exists():
        rows.append({"Model": model_name, "Accuracy": "missing dir", "Questions": 0, "Avg Steps": "-", "Avg Input Tokens": "-", "Avg Output Tokens": "-", "Avg Tokens": "-", "Errors": "-"})
        continue
    stats = summarize_results(pkl_dir)
    if stats is None:
        rows.append({"Model": model_name, "Accuracy": "no data", "Questions": 0, "Avg Steps": "-", "Avg Input Tokens": "-", "Avg Output Tokens": "-", "Avg Tokens": "-", "Errors": "-"})
        continue
    rows.append({
        "Model": model_name,
        "Accuracy": f"{stats['correct']}/{stats['n']} = {stats['accuracy']:.1%}",
        "Questions": stats["n"],
        "Avg Steps": f"{stats['avg_steps']:.1f}",
        "Avg Input Tokens": f"{stats['avg_input_tokens']:,.0f}",
        "Avg Output Tokens": f"{stats['avg_output_tokens']:,.0f}",
        "Avg Tokens": f"{stats['avg_tokens']:,.0f}",
        "Errors": stats["errors"],
    })

pd.DataFrame(rows)

,Model,Accuracy,Questions,Avg Steps,Avg Input Tokens,Avg Output Tokens,Avg Tokens,Errors
0,gpt-4o,43/165 = 26.1%,165,10.8,"134,996","1,609","136,605",0
1,gpt-5.4-mini,54/165 = 32.7%,165,5.4,"36,684",607,"37,291",0
2,Qwen3.7-Plus,103/165 = 62.4%,165,19.5,"452,400","3,028","455,429",1
3,Qwen3.5-9B,83/165 = 50.3%,165,18.9,"463,766","4,031","467,797",0


Based on the above results, it does seem that open-source models use more tokens and more steps but are performing better than closed source models. So maybe we should simply shift to open-source models.

## Markov ReAct

Another dimension that we wanted to test was how independent is each iteration of ReAct in comparison to the previous ones. Can the previous iterations be completely pruned and how much would this impact the performance?

In [14]:
from IPython.display import display

MARKOV_DIR = Path("markovReAct")

# Reuses summarize_results and MODEL_DIRS (naive baselines) defined in the Baselines cell above.
CONFIG_DIRS = {
    "gpt-4o": {
        "naive": MODEL_DIRS["gpt-4o"],
        "w1": MARKOV_DIR / "markov_react_gpt-4o_w1",
        "w3": MARKOV_DIR / "markov_react_gpt-4o_w3",
        "w5": MARKOV_DIR / "markov_react_gpt-4o_w5",
    },
    "gpt-5.4-mini": {
        "naive": MODEL_DIRS["gpt-5.4-mini"],
        "w1": MARKOV_DIR / "markov_react_gpt-5.4-mini_w1",
        "w3": MARKOV_DIR / "markov_react_gpt-5.4-mini_w3",
        "w5": MARKOV_DIR / "markov_react_gpt-5.4-mini_w5",
    },
    "Qwen3.7-Plus": {
        "naive": MODEL_DIRS["Qwen3.7-Plus"],
        "w1": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.7-Plus_w1",
        "w3": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.7-Plus_w3",
        "w5": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.7-Plus_w5",
    },
    "Qwen3.5-9B": {
        "naive": MODEL_DIRS["Qwen3.5-9B"],
        "w1": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.5-9B_w1",
        "w3": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.5-9B_w3",
        "w5": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.5-9B_w5",
    },
}
CONFIG_ORDER = ["naive", "w1", "w3", "w5"]

rows = []
for model_name, configs in CONFIG_DIRS.items():
    for config_name, pkl_dir in configs.items():
        if not pkl_dir.exists():
            rows.append({"Model": model_name, "Config": config_name, "Accuracy": None, "Avg Steps": None, "Tokens (in/out/total)": None})
            continue
        stats = summarize_results(pkl_dir)
        if stats is None:
            rows.append({"Model": model_name, "Config": config_name, "Accuracy": None, "Avg Steps": None, "Tokens (in/out/total)": None})
            continue
        rows.append({
            "Model": model_name,
            "Config": config_name,
            "Accuracy": f"{stats['accuracy']:.1%}",
            "Avg Steps": round(stats["avg_steps"], 1),
            "Tokens (in/out/total)": f"{stats['avg_input_tokens']:,.0f} / {stats['avg_output_tokens']:,.0f} / {stats['avg_tokens']:,.0f}",
        })

results_df = pd.DataFrame(rows)

for metric in ["Accuracy", "Avg Steps", "Tokens (in/out/total)"]:
    print(f"=== {metric} ===")
    display(results_df.pivot(index="Model", columns="Config", values=metric)[CONFIG_ORDER])

=== Accuracy ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,50.3%,29.1%,33.9%,44.2%
Qwen3.7-Plus,62.4%,29.7%,54.5%,53.9%
gpt-4o,26.1%,28.5%,28.5%,30.9%
gpt-5.4-mini,32.7%,27.9%,33.9%,30.9%


=== Avg Steps ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,18.9,30.0,26.6,24.0
Qwen3.7-Plus,19.5,32.6,24.7,22.4
gpt-4o,10.8,21.0,13.4,11.9
gpt-5.4-mini,5.4,8.6,5.2,5.0


=== Tokens (in/out/total) ===


Config,naive,w1,w3,w5
Model,,,,
Qwen3.5-9B,"463,766 / 4,031 / 467,797","127,817 / 5,344 / 133,161","183,237 / 5,284 / 188,521","218,227 / 4,544 / 222,771"
Qwen3.7-Plus,"452,400 / 3,028 / 455,429","133,445 / 3,706 / 137,151","168,823 / 3,336 / 172,159","206,597 / 3,088 / 209,685"
gpt-4o,"134,996 / 1,609 / 136,605","81,046 / 2,650 / 83,697","79,174 / 1,954 / 81,128","86,789 / 1,618 / 88,407"
gpt-5.4-mini,"36,684 / 607 / 37,291","31,276 / 986 / 32,263","24,328 / 563 / 24,891","27,341 / 577 / 27,918"


The above results are very interesting as they show how the number of steps are more in Markov ReAct but the token counts are low. It turns out the source of low token counts is low input tokens. And in fact, the output tokens are higher in Markov ReAct than naive ReAct.

It is also worth noting that for gpt-4o and gpt-5.4-mini, naiveReAct is not the most accurate version which indicates to possibility of improving performance when the context is cleverly chosen.